In [5]:
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 7.9 MB/s eta 0:00:00


In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker()

random.seed(42)
np.random.seed(42)

In [9]:
bases = [
    "Bangalore",
    "Mysuru",
    "Hyderabad",
    "Chennai",
    "Pune"
]
aircraft_types = ["C172", "PA28", "DA40"]

aircraft_data = []

for i in range(25):

    aircraft_id = f"A{i+1:03}"

    aircraft_type = random.choice(aircraft_types)

    available_hours = random.randint(120, 220)

    # Different aircraft behave differently
    if aircraft_type == "DA40":
        downtime = random.randint(40, 90)
        defects = random.randint(6, 15)

    elif aircraft_type == "C172":
        downtime = random.randint(10, 40)
        defects = random.randint(1, 6)

    else:
        downtime = random.randint(20, 60)
        defects = random.randint(3, 10)

    aircraft_data.append({
        "aircraft_id": aircraft_id,
        "registration": fake.bothify("VT-???"),
        "type": aircraft_type,
        "base_id": random.choice(bases),
        "total_available_hours": available_hours,
        "maintenance_downtime_hours": downtime,
        "defect_count": defects
    })

aircraft_df = pd.DataFrame(aircraft_data)

In [10]:
instructors = []

for i in range(30):

    qualified = random.choice(aircraft_types)

    duty_hours = random.randint(80, 220)

    flight_hours = int(duty_hours * random.uniform(0.4, 0.8))

    instructors.append({
        "instructor_id": f"I{i+1:03}",
        "name": fake.name(),
        "base_id": random.choice(bases),
        "aircraft_qualified": qualified,
        "total_duty_hours": duty_hours,
        "total_flight_hours": flight_hours
    })

instructors_df = pd.DataFrame(instructors)

In [11]:
personas = [
    "excellent",
    "financial_risk",
    "study_risk",
    "operational_risk",
    "high_risk"
]
cadets = []

for i in range(200):

    cadet_id = f"C{i+1:03}"

    course = random.choice(["PPL", "CPL"])

    required_hours = 45 if course == "PPL" else 200

    persona = random.choice(personas)

    # PERSONA LOGIC
    if persona == "excellent":
        flown = required_hours * random.uniform(0.6, 0.95)

    elif persona == "high_risk":
        flown = required_hours * random.uniform(0.05, 0.25)

    else:
        flown = required_hours * random.uniform(0.2, 0.7)

    cadets.append({
        "cadet_id": cadet_id,
        "name": fake.name(),
        "course": course,
        "home_base": random.choice(bases),
        "total_required_hours": required_hours,
        "total_flown_hours": round(flown, 1),
        "persona": persona,
        "enrollment_date":
            fake.date_between(
                start_date='-2y',
                end_date='today'
            )
    })

cadets_df = pd.DataFrame(cadets)

In [12]:
statuses = ["completed", "delayed", "cancelled"]

status_probs = [0.72, 0.18, 0.10]
sorties = []

lesson_types = [
    "Navigation",
    "Circuit",
    "General Handling",
    "Cross Country",
    "Night Flying"
]

cancel_reasons = [
    "Weather",
    "Aircraft Defect",
    "Instructor Unavailable",
    "ATC Restriction"
]

start_date = datetime(2025, 1, 1)

for i in range(12000):

    cadet = cadets_df.sample(1).iloc[0]

    instructor = instructors_df.sample(1).iloc[0]

    aircraft = aircraft_df[
        aircraft_df["type"] ==
        instructor["aircraft_qualified"]
    ].sample(1).iloc[0]

    scheduled_start = start_date + timedelta(
        days=random.randint(0, 180),
        hours=random.randint(5, 18)
    )

    scheduled_end = scheduled_start + timedelta(hours=1.5)

    status = np.random.choice(
        statuses,
        p=status_probs
    )

    delay = 0
    cancel_reason = None

    if status == "completed":

        delay = random.randint(0, 40)

        actual_start = scheduled_start + timedelta(minutes=delay)

        actual_end = actual_start + timedelta(hours=1.5)

    elif status == "delayed":

        delay = random.randint(30, 120)

        actual_start = scheduled_start + timedelta(minutes=delay)

        actual_end = actual_start + timedelta(hours=1.5)

    else:
        actual_start = None
        actual_end = None

        cancel_reason = np.random.choice(
            cancel_reasons,
            p=[0.55, 0.25, 0.1, 0.1]
        )

    sorties.append({
        "sortie_id": f"S{i+1:05}",
        "cadet_id": cadet["cadet_id"],
        "instructor_id": instructor["instructor_id"],
        "aircraft_id": aircraft["aircraft_id"],
        "base_id": cadet["home_base"],
        "scheduled_start": scheduled_start,
        "scheduled_end": scheduled_end,
        "actual_start": actual_start,
        "actual_end": actual_end,
        "status": status,
        "delay_minutes": delay,
        "cancel_reason": cancel_reason,
        "lesson_type": random.choice(lesson_types)
    })

sorties_df = pd.DataFrame(sorties)

In [13]:
subjects = [
    "Navigation",
    "Meteorology",
    "Air Regulations",
    "RTR",
    "Technical General"
]

study = []

for _, cadet in cadets_df.iterrows():

    for subject in random.sample(subjects, 3):

        persona = cadet["persona"]

        if persona == "excellent":
            quiz = random.randint(75, 95)
            progress = random.uniform(0.7, 1.0)

        elif persona == "high_risk":
            quiz = random.randint(30, 55)
            progress = random.uniform(0.1, 0.4)

        else:
            quiz = random.randint(50, 80)
            progress = random.uniform(0.3, 0.8)

        total_chapters = random.randint(20, 40)

        completed = int(
            total_chapters * progress
        )

        study.append({
            "cadet_id": cadet["cadet_id"],
            "subject": subject,
            "chapters_completed": completed,
            "total_chapters": total_chapters,
            "avg_quiz_score": quiz,
            "last_active_date":
                fake.date_between(
                    start_date='-90d',
                    end_date='today'
                ),
            "practice_tests_attempted":
                random.randint(0, 10)
        })

study_df = pd.DataFrame(study)

In [16]:
payments = []

for _, cadet in cadets_df.iterrows():

    persona = cadet["persona"]

    total = random.randint(150000, 500000)

    if persona == "financial_risk":
        paid = total * random.uniform(0.3, 0.6)

    else:
        paid = total * random.uniform(0.7, 1.0)

    payments.append({
        "cadet_id": cadet["cadet_id"],
        "invoiced_amount": int(total),
        "paid_amount": int(paid),
        "outstanding_amount": int(total - paid),
        "last_payment_date":
            fake.date_between(
                start_date='-120d',
                end_date='today'
            )
    })

payments_df = pd.DataFrame(payments)

In [14]:
# completed sortie with missing actual time

idx = sorties_df.sample(20).index

sorties_df.loc[idx, "actual_start"] = None

In [17]:
# invalid payment calculation

idx = payments_df.sample(10).index

payments_df.loc[idx, "outstanding_amount"] += 5000

In [18]:
# aircraft downtime > available

idx = aircraft_df.sample(3).index

aircraft_df.loc[idx,
    "maintenance_downtime_hours"] = 400

In [19]:
aircraft_df.to_csv("aircraft.csv", index=False)

cadets_df.to_csv("cadets.csv", index=False)

instructors_df.to_csv("instructors.csv", index=False)

sorties_df.to_csv("sorties.csv", index=False)

study_df.to_csv("toga_study.csv", index=False)

payments_df.to_csv("payments.csv", index=False)